In [1]:
import xarray as xr
import glob

files = sorted(glob.glob("/scratch/benbar/NAARC/Initial_Conditions/IC_compressed_*.nc"))
datasets = [xr.open_dataset(f, chunks={}) for f in files]

In [2]:
datasets[0]

<xarray.Dataset> Size: 584MB
Dimensions:  (z: 75, y: 901, x: 1080)
Dimensions without coordinates: z, y, x
Data variables:
    toce     (z, y, x) float32 292MB dask.array<chunksize=(1, 721, 1080), meta=np.ndarray>
    soce     (z, y, x) float32 292MB dask.array<chunksize=(1, 721, 1080), meta=np.ndarray>

In [3]:
rows = []

for iy in range(4):
    row = xr.concat(
        [datasets[iy * 4 + ix] for ix in range(4)],
        dim="x"
    )
    rows.append(row)

ds_full = xr.concat(rows, dim="y")

In [4]:
ds_full

<xarray.Dataset> Size: 9GB
Dimensions:  (z: 75, y: 3605, x: 4320)
Dimensions without coordinates: z, y, x
Data variables:
    toce     (z, y, x) float32 5GB dask.array<chunksize=(1, 721, 1080), meta=np.ndarray>
    soce     (z, y, x) float32 5GB dask.array<chunksize=(1, 721, 1080), meta=np.ndarray>

In [7]:
ds_full = ds_full.chunk({'z': 1, 'y': 721, 'x': 1080})

In [11]:
fill_value = 1e20
ds_full = ds_full.fillna(fill_value)

encoding = {
    var: {
        "zlib": True,
        "complevel": 1,
        "_FillValue": fill_value,
        "dtype": "float32",
        "chunksizes": (1, 721, 1080),  # (deptht, y, x) chunk shape on disk
    }
    for var in ds_full.data_vars
}

ds_full.to_netcdf(
    "/scratch/benbar/NAARC/Initial_Conditions/IC_compressed.nc",
    mode="w",
    format="NETCDF4",
    engine="netcdf4",
    encoding=encoding,
)